# 04: Validate the monthly recovery table

**Goal:** Check `recovery_monthly.csv` using pandas.

**Input:** `data/processed/recovery_monthly.csv` (from notebook 03).

**Output:** None.

**Steps:** Load the `recovery_monthly.csv`, check its structure and values, review the entire series.

**Run from the repo root.**

## Setup and load

The data is loaded into a dataframe called `recovery`.

In [1]:
import pandas as pd

CSV_PATH = "data/processed/recovery_monthly.csv"
CLEANED_PATH = "data/processed/trimet_monthly.csv"

recovery = pd.read_csv(CSV_PATH, parse_dates=["month"])

## Check the load shape and data types

- Expect `(186, 13)`.
- Expect `month` as datetime, `mode` and `mode_name` as strings, and all other columns as floats.

In [2]:
print(recovery.shape)
print(recovery.dtypes)
display(recovery.head())

(186, 13)
mode                            object
mode_name                       object
month                   datetime64[ns]
upt                            float64
vrh                            float64
upt_recovery_monthly           float64
vrh_recovery_monthly           float64
upt_rolling                    float64
vrh_rolling                    float64
upt_recovery_rolling           float64
vrh_recovery_rolling           float64
upt_per_vrh_monthly            float64
upt_per_vrh_rolling            float64
dtype: object


,mode,mode_name,month,upt,vrh,upt_recovery_monthly,vrh_recovery_monthly,upt_rolling,vrh_rolling,upt_recovery_rolling,vrh_recovery_rolling,upt_per_vrh_monthly,upt_per_vrh_rolling
0,LR,MAX light rail,2019-01-01,3201280.0,53964.0,1.0,1.0,NaN,NaN,NaN,NaN,59.322511,NaN
1,LR,MAX light rail,2019-02-01,2861440.0,49134.0,1.0,1.0,NaN,NaN,NaN,NaN,58.237473,NaN
2,LR,MAX light rail,2019-03-01,3197550.0,54111.0,1.0,1.0,NaN,NaN,NaN,NaN,59.092421,NaN
3,LR,MAX light rail,2019-04-01,3209820.0,53024.0,1.0,1.0,NaN,NaN,NaN,NaN,60.535229,NaN
4,LR,MAX light rail,2019-05-01,3429260.0,54476.0,1.0,1.0,NaN,NaN,NaN,NaN,62.949923,NaN


## Check structure

- Expect 186 rows and the 13 columns in order.
- Expect only the modes LR and MB. 

In [3]:
EXPECTED_COLS = [
    "mode",
    "mode_name",
    "month",
    "upt",
    "vrh",
    "upt_recovery_monthly",
    "vrh_recovery_monthly",
    "upt_rolling",
    "vrh_rolling",
    "upt_recovery_rolling",
    "vrh_recovery_rolling",
    "upt_per_vrh_monthly",
    "upt_per_vrh_rolling",
]

assert recovery.shape == (186, 13)
assert list(recovery.columns) == EXPECTED_COLS
assert set(recovery["mode"]) == {"LR", "MB"}

print("Structure OK")

Structure OK


## Check months

- Expect one row per mode and month combination.
- Expect 93 months per mode January 2019 to September 2026.

In [4]:
assert dict(recovery.groupby("mode")["month"].nunique()) == {"LR": 93, "MB": 93}
assert not recovery.duplicated(["mode", "month"]).any()
assert (recovery["month"].dt.day == 1).all()

first_months = recovery.groupby("mode")["month"].min()
last_months = recovery.groupby("mode")["month"].max()

assert (first_months == "2019-01-01").all()
assert (last_months == "2026-09-01").all()

print("Months OK")

Months OK


## Check for unexpected nulls

- Expected nulls are the 5 rolling columns. There should be 22 per column, all before December 2019.
- No other columns should have null values.

In [5]:
display(recovery.isna().sum())

EXPECTED_NULL_COLS = [
    "upt_rolling",
    "vrh_rolling",
    "upt_recovery_rolling",
    "vrh_recovery_rolling",
    "upt_per_vrh_rolling",
]

assert (recovery[EXPECTED_NULL_COLS].isna().sum() == 22).all()
assert recovery.drop(columns=EXPECTED_NULL_COLS).notna().all().all()

assert (
    recovery.loc[recovery["month"] < "2019-12-01", EXPECTED_NULL_COLS]
    .isna()
    .all()
    .all()
)

assert (
    recovery.loc[recovery["month"] >= "2019-12-01", EXPECTED_NULL_COLS]
    .notna()
    .all()
    .all()
)

print("Nulls OK")

mode                     0
mode_name                0
month                    0
upt                      0
vrh                      0
upt_recovery_monthly     0
vrh_recovery_monthly     0
upt_rolling             22
vrh_rolling             22
upt_recovery_rolling    22
vrh_recovery_rolling    22
upt_per_vrh_monthly      0
upt_per_vrh_rolling     22
dtype: int64

Nulls OK


## Check recovery values

- Expect upt and vrh to be greater than 0.
- Expect recovery values to be between 0 and 1.5.
- Expect baseline values to be 1.0 (2019 monthly recovery values should be exactly 1.0. December 2019 rolling recovery should also be 1.0.)


In [6]:
assert (recovery["upt"] > 0).all()
assert (recovery["vrh"] > 0).all()

MONTHLY_RECOVERY_COLS = [
    "upt_recovery_monthly",
    "vrh_recovery_monthly",
]
ROLLING_RECOVERY_COLS = [
    "upt_recovery_rolling",
    "vrh_recovery_rolling",
]

rows_2019 = recovery["month"] < "2020-01-01"
dec_2019 = recovery["month"] == "2019-12-01"

assert (recovery[MONTHLY_RECOVERY_COLS] >= 0).all().all()
assert (recovery[MONTHLY_RECOVERY_COLS] <= 1.5).all().all()

assert (recovery[ROLLING_RECOVERY_COLS].dropna() >= 0).all().all()
assert (recovery[ROLLING_RECOVERY_COLS].dropna() <= 1.5).all().all()

monthly_2019_diff = (
    (recovery.loc[rows_2019, MONTHLY_RECOVERY_COLS] - 1).abs().max().max()
)
dec_2019_diff = (recovery.loc[dec_2019, ROLLING_RECOVERY_COLS] - 1).abs().max().max()
assert monthly_2019_diff < 1e-9 and dec_2019_diff < 1e-9

print("Recovery values OK")

Recovery values OK


## Check derived values

- Recheck trips per revenue hour columns
- Expect `upt_per_vrh_monthly` = `upt` / `vrh` for each month.
- Expect `upt_per_vrh_rolling` = `upt_rolling` / `vrh_rolling` for each month.

In [7]:
monthly_ratio_diff = (
    (recovery["upt"] / recovery["vrh"] - recovery["upt_per_vrh_monthly"]).abs().max()
)
rolling_ratio_diff = (
    (
        recovery["upt_rolling"] / recovery["vrh_rolling"]
        - recovery["upt_per_vrh_rolling"]
    )
    .abs()
    .max()
)

assert monthly_ratio_diff < 1e-9 and rolling_ratio_diff < 1e-9

print("Derived values OK")

Derived values OK


## Check against cleaned data

- Expect `upt` and `vrh` to match the values in `trimet_monthly.csv` for each MB and LR month.

In [8]:
cleaned = pd.read_csv(CLEANED_PATH, parse_dates=["month"])
cleaned = cleaned[cleaned["mode"].isin(["MB", "LR"])]

compare = recovery.merge(
    cleaned[["mode", "month", "upt", "vrh"]],
    on=["mode", "month"],
    suffixes=("", "_cleaned"),
    validate="one_to_one",
)

assert len(compare) == len(recovery)
assert (compare["upt"] == compare["upt_cleaned"]).all()
assert (compare["vrh"] == compare["vrh_cleaned"]).all()

print("Values match cleaned data")

Values match cleaned data


**Check results:** Structure, months, nulls, value ranges, the 2019 baseline, derived values, and inputs against the cleaned data all pass.

## Review the whole series

- Checks the range of each value, highest and lowest months, unusual months, and a year by year view of recovery. 

### Range of recovery values

- Shows the maximum and minimum of each recovery value

In [9]:
RECOVERY_COLS = MONTHLY_RECOVERY_COLS + ROLLING_RECOVERY_COLS

display(recovery.groupby("mode")[RECOVERY_COLS].agg(["min", "max"]).round(3))

upt_recovery_monthly        vrh_recovery_monthly         \
                      min    max                  min    max   
mode                                                           
LR                  0.318  1.061                0.619  1.036   
MB                  0.308  1.106                0.813  1.050   

     upt_recovery_rolling        vrh_recovery_rolling         
                      min    max                  min    max  
mode                                                          
LR                  0.374  1.002                0.836  1.004  
MB                  0.401  1.010                0.845  1.009

### Highest and lowest months

- Shows the months with the highest and lowest recovery values for each mode

In [10]:
for col in ["upt_recovery_monthly", "vrh_recovery_monthly"]:
    print("Highest for", col)
    display(
        recovery.loc[recovery.groupby("mode")[col].idxmax()][["mode", "month", col]]
    )
    print("Lowest for", col)
    display(
        recovery.loc[recovery.groupby("mode")[col].idxmin()][["mode", "month", col]]
    )

Highest for upt_recovery_monthly


,mode,month,upt_recovery_monthly
13,LR,2020-02-01,1.061238
106,MB,2020-02-01,1.105973


Lowest for upt_recovery_monthly


,mode,month,upt_recovery_monthly
16,LR,2020-05-01,0.317809
108,MB,2020-04-01,0.307609


Highest for vrh_recovery_monthly


,mode,month,vrh_recovery_monthly
13,LR,2020-02-01,1.036248
106,MB,2020-02-01,1.049508


Lowest for vrh_recovery_monthly


,mode,month,vrh_recovery_monthly
61,LR,2024-02-01,0.619042
138,MB,2022-10-01,0.813317


### Unusual months

- List months where UPT recovery > 1.0 (more trips than same month in 2019).
- List months where VRH recovery < 0.8 (under 80% of same month in 2019).
- Not necessarily an error, but worth looking into.

In [11]:
unusual = recovery[
    (recovery["upt_recovery_monthly"] > 1) | (recovery["vrh_recovery_monthly"] < 0.8)
]

display(
    unusual[["mode", "month", "upt_recovery_monthly", "vrh_recovery_monthly"]].round(3)
)

,mode,month,upt_recovery_monthly,vrh_recovery_monthly
13,LR,2020-02-01,1.061,1.036
60,LR,2024-01-01,0.446,0.639
61,LR,2024-02-01,0.540,0.619
105,MB,2020-01-01,1.021,1.026
106,MB,2020-02-01,1.106,1.050


### Yearly recovery

- Look at the rolling recovery for December of each year.
- Also include September 2026 (latest available data).

In [12]:
display(
    recovery[(recovery["month"].dt.month == 12) | (recovery["month"] == "2026-09-01")][
        ["mode", "month", "upt_recovery_rolling", "vrh_recovery_rolling"]
    ].round(3)
)

,mode,month,upt_recovery_rolling,vrh_recovery_rolling
11,LR,2019-12-01,1.000,1.000
23,LR,2020-12-01,0.499,0.949
35,LR,2021-12-01,0.418,0.922
47,LR,2022-12-01,0.535,0.912
59,LR,2023-12-01,0.603,0.887
71,LR,2024-12-01,0.597,0.870
83,LR,2025-12-01,0.575,0.896
92,LR,2026-09-01,0.573,0.903
104,MB,2019-12-01,1.000,1.000
116,MB,2020-12-01,0.527,0.930


## Observations

- The lowest monthly UPT recovery for both modes occurs in spring 2020 (LR in May at 0.318, MB in April at 0.308).
- As of September 2026, rolling UPT recovery is at 0.573 for LR and 0.762 for MB.
- As of September 2026, rolling VRH recovery is at 0.903 for LR and 0.948 for MB (higher than UPT for both modes).
- LR's December rolling UPT recovery was highest in 2023 (0.603) and lower in each December since (0.597 in 2024, 0.575 in 2025).
- The rolling UPT recovery for MB has risen every year since December 2021 (0.486 in 2021 to 0.752 in 2025).
- LR has two months where monthly VRH recovery was under 0.8: January and February 2024 (at 0.639 and 0.619 respectively).
- MB has no months where VRH recovery was under 0.8.